# Qiskit Fall Fest 2026 — Phase 1: Problem S3
## Controlled Architecture Benchmarking: Processor A vs. Processor B
**Problem Statement:** S3 — Noise in Trotter Simulation  
**Track:** Quantum Simulation  

This notebook provides a complete, reproducible execution of the controlled quantum simulation experiment comparing **Processor A (5Q Line Baseline)** and **Processor B (7Q Heavy-Hex Baseline)**.

In [ ]:
# 1. Environment & Imports
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from qiskit_aer import AerSimulator

from src.processors import load_processor_config, build_coupling_map, build_noise_model
from src.problem import TrotterSimulationModel
from src.transpilation import compile_for_processor, extract_circuit_metrics
from src.metrics import compute_magnetization_from_counts, compute_tvd, compute_classical_fidelity
from src.visualization import plot_processor_topology, plot_logical_vs_transpiled, plot_ab_performance, plot_architecture_tradeoffs

print("All modules imported successfully!")

### 2. Processor Architecture Loading & Topology Inspection

In [ ]:
proc_a_cfg = load_processor_config("processors/processor_A.json")
proc_b_cfg = load_processor_config("processors/processor_B.json")

cmap_a = build_coupling_map(proc_a_cfg)
cmap_b = build_coupling_map(proc_b_cfg)

noise_a = build_noise_model(proc_a_cfg)
noise_b = build_noise_model(proc_b_cfg)

print(f"Loaded {proc_a_cfg['display_name']} ({proc_a_cfg['num_qubits']} qubits)")
print(f"Loaded {proc_b_cfg['display_name']} ({proc_b_cfg['num_qubits']} qubits)")

# Plot coupling maps
plot_processor_topology(proc_a_cfg, "results/figures/processor_A_graph.png")
plot_processor_topology(proc_b_cfg, "results/figures/processor_B_graph.png")
print("Topologies plotted and saved in results/figures/")

### 3. Initialize Problem S3: 4-Spin Transverse-Field Ising Model (TFIM)
$$H = - J \sum_{i=0}^{N-2} Z_i Z_{i+1} - J_{\text{cross}} Z_0 Z_{N-1} - h \sum_{i=0}^{N-1} X_i$$
Initial state: $|1010\rangle$, Evolution time: $T = 1.5$.

In [ ]:
num_spins = 4
total_time = 1.5
shots = 4096
seed = 42

model = TrotterSimulationModel(
    num_spins=num_spins,
    J=1.0,
    h=0.8,
    J_cross=0.5,
    initial_state_str="1010"
)

exact_mag = model.compute_exact_magnetization(total_time)
print(f"Exact analytical <M_z(T={total_time})> = {exact_mag:.4f}")

### 4. Execute Benchmark Pipeline
Run `main.py` directly to populate results and generate final comparison artifacts.

In [ ]:
import main
main.run_experiment()

### 5. Display Benchmark Summary Table

In [ ]:
df_summary = pd.read_csv("results/tables/final_comparison.csv")
df_summary